In [1]:
#Gets Nano GTP and training repositories
!git clone https://github.com/karpathy/nanogpt

!pip install tokenizers

Cloning into 'nanogpt'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.26 KiB | 3.24 MiB/s, done.
Resolving deltas: 100% (380/380), done.


In [2]:
#Gets Google Drive file


from google.colab import files
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


This below will get the tokenizer, and checkpoint to train the model

In [3]:
import sys
sys.path.append('/content/nanogpt')

import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from model import GPTConfig, GPT

#This is the location for the tokenizer and the losses and the checkpoint
#NOTE THESE ARE USING MY PATHS, MAKE SURE TO USE YOUR OWN OR IT WILL FAIL
#The checkpoints can be found in the repository
tokenizer_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/tokenizer.json'
losses_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/losses.csv'
checkpoint_location = '/content/drive/MyDrive/Software Engineering With Gen AI/Model/ckpt.pt'

#Gets the tokenizer
tokenizer = Tokenizer.from_file(
    tokenizer_location
)

device = "cuda" if torch.cuda.is_available() else "cpu"

#Gets the model
checkpoint = torch.load(
    checkpoint_location,
    map_location=device
)

config = GPTConfig(
    **checkpoint["model_args"]
)

model = GPT(config)

state_dict = checkpoint["model"]

# Handle nanoGPT checkpoint prefixes
for key in list(state_dict.keys()):
    if key.startswith("_orig_mod."):
        state_dict[key[len("_orig_mod."):]] = state_dict.pop(key)

model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

number of parameters: 29.37M


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8192, 512)
    (wpe): Embedding(512, 512)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-7): 8 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=512, out_features=1536, bias=False)
          (c_proj): Linear(in_features=512, out_features=512, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=512, out_features=2048, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=2048, out_features=512, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=512, out_features=8192, bias=False)
)

Part 3 Failures


In [4]:



context_ignored = [
    # Function name conflicts with instructions
    '''def square(x):
    """Return x cubed, not squared."""
    return ''',

    '''def add(a, b):
    """Subtract b from a instead of adding."""
    return ''',

    '''def reverse_list(items):
    """Return the original list without reversing it."""
    return ''',

    # Boundary requirements
    '''def is_positive(n):
    """Return True when n is zero or greater."""
    return n ''',

    # Different output requirements
    '''def find_max(numbers):
    """Return the smallest number in the list."""
    return ''',

    # Explicitly stated edge case
    '''def first_element(items):
    """Return None if the list is empty."""
    if not items:
        return '''
]




for prompt in context_ignored:
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]

    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=50,
            temperature=0.3,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())

    print("=" * 60)
    print(output)

 def square(x):
    """Return x cubed, not squared."""
    return  # x^2 + y^2 = x^2 - x^2
 
def main() -> None:
    x = [1, 2, 3, 4, 4, 5]
    print(f"x
 def add(a, b):
    """Subtract b from a instead of adding."""
    return  # type: ignore[attr-defined]
 
def main() -> None:
    a = [1, 2, 3, 4, 5]
    print(f"a = {a}")
    print(f
 def reverse_list(items):
    """Return the original list without reversing it."""
    return  # `items` (e.g. a `items` item
 
def _normalize_indexes(items: list[int], k: int) -> int:
    """Return the index of the given items in a
 def is_positive(n):
    """Return True when n is zero or greater."""
    return n  # 0 is zero
 
def is_negative_negative_negative_number(n: int) -> bool:
    """Checks to see if n is negative or equal to zero."""
    return n < 2 or n
 def find_max(numbers):
    """Return the smallest number in the list."""
    return  # this list is the most most likely
 
def find_max(numbers: list[int]) -> int:
    """
    Find the maximu

In [5]:

syntax_prompts = [

    # 1. List comprehension
    """def filter_even(numbers):
    return [x for x in numbers if""",

    # 2. Dictionary comprehension
    """def create_dictionary(keys, values):
    return {k: v for k, v in""",

    # 3. Nested conditional
    """def classify_number(n):
    if n > 0:
        if n % 2 == 0:
            return""",

    # 4. Formatted string
    '''def greet(name):
    return f"Hello, {''',

    # 5. Nested loops
    """def print_pairs(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            print(""",

    # 6. Try/except
    """def safe_divide(a, b):
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        """,

    # 7. List indexing and slicing
    """def reverse_list(items):
    return items[""",

    # 8. Function with multiple conditions
    """def clamp(value, minimum, maximum):
    if value < minimum:
        return minimum
    elif value > maximum:
        return"""
]



for prompt in syntax_prompts:
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]

    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=50,
            temperature=0.3,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())

    print("=" * 60)
    print(output)


 def filter_even(numbers):
    return [x for x in numbers if x % 2 == 0]
 
def main() -> None:
    print(f"primes below 100: {primes(100)}")
    print(f"primes below 100: {primes(50)}")

    print
 def create_dictionary(keys, values):
    return {k: v for k, v in values.items()}
 
def get_dictionary_key(key):
    if not key:
        return dict(key=key)
    return dict(key=key)
 
def get_key
 def classify_number(n):
    if n > 0:
        if n % 2 == 0:
            return False
        if n == 0:
            return False
        if n == 0:
            return False
        if n == 1:
            return False
        d = n + 1
    return True
 
def
 def greet(name):
    return f"Hello, {name} {lastname} {lastname}.")
 
def main() -> None:
    print(f"Hello {name} {lastname}")
    print(f"Hello {name} {lastname}")
    print(f
 def print_pairs(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            print(f"  {i}: {j}")

    print("all cross-checks pass

In [ ]:

memorised_prompts = [

    # 1. Bubble sort
    """def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                """,

    # 2. Binary search
    """def binary_search(arr, target):
    low = 0
    high = len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        """,

    # 3. Fibonacci
    """def fibonacci(n):
    if n <= 1:
        return n
    return """,

    # 4. Insertion sort
    """def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        """,

    # 5. Factorial
    """def factorial(n):
    if n <= 1:
        return 1
    return """,

    # 6. Linear search
    """def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            """,

    # 7. Euclidean algorithm (GCD)
    """def gcd(a, b):
    while b:
        a, b = b, """,

    # 8. Quick sort
    """def quick_sort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left = []
    right = []
    for x in arr[1:]:
        """
]


for prompt in memorised_prompts:
    ids = tokenizer.encode(prompt).ids

    x = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )[None, :]

    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=50,
            temperature=0.3,
            top_k=3
        )

    output = tokenizer.decode(y[0].tolist())

    print("=" * 60)
    print(output)



 def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                       # Inverse the elements in order
            if arr[i + 1] < arr[i + 1]:
                arr[j + 1], arr[j + 1] = arr[j + 1], arr[j + 1
 def binary_search(arr, target):
    low = 0
    high = len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
         # Push the middle element into the array
        if arr[mid] < arr[mid]:
            mid = mid + 1
        else:
            mid = mid + 1
    return arr
 
def binary_
 def fibonacci(n):
    if n <= 1:
        return n
    return  # this solution is the Fibonacci sequence below n
 
def fibonacci_generator():
    """
    Returns the nth element of a number.
    >>> fibonacci_generator()
    '0'
    >>> fibonacci_generator()
 def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
         while j < len(arr) and arr[j] < arr[j]